# **Problem 4**

You will need to complete this problem on an alternative platform, such as Google Colab. Once you are finished, run all cells and show all outputs. You can download the notebook as a PDF and append it to the rest of your homework submission.

This notebook builds a network for 2D convolution.  It works with the MNIST dataset, which was the original classic dataset for classifying images.  The network will take a 28x28 grayscale image and classify it into one of 10 classes representing a digit.

If you are using Google Colab, you can change your runtime to an instance with GPU support to speed up training, e.g. a T4 GPU.

In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [42]:
# Run this once to load the train and test data into a dataloader class
# that will provide the batches.

batch_size_train = 64
batch_size_test = 1000

# Image transformations
transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.1307,), (0.3081,))])

# Get datasets and convert to dataloaders
train_loader = torch.utils.data.DataLoader(
  datasets.MNIST(root='data', train=True, download=True, transform=transform), 
  batch_size=batch_size_train, shuffle=True)

test_loader = torch.utils.data.DataLoader(
  datasets.MNIST(root='data', train=False, download=True, transform=transform), 
  batch_size=batch_size_test, shuffle=True)

In [43]:
# Let's draw some of the training data
examples = enumerate(test_loader)
batch_idx, (example_data, example_targets) = next(examples)

fig = plt.figure()
for i in range(6):
    plt.subplot(2,3,i+1)
    plt.tight_layout()
    plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
    plt.title("Ground Truth: {}".format(example_targets[i]))
    plt.xticks([])
    plt.yticks([])
plt.show()

## Part 1 (6 points)

Define the neural network by defining a class, and then defining the parameters in the constructor. Then we use a function called forward to actually run the network.

In [44]:
# TODO Change this class to implement
# 1. A valid convolution with kernel size 5, 1 input channel and 10 output channels
# 2. A max pooling operation over a 2x2 area
# 3. A Relu
# 4. A valid convolution with kernel size 5, 10 input channels and 20 output channels
# 5. A 2D Dropout layer
# 6. A max pooling operation over a 2x2 area
# 7. A relu
# 8. A flattening operation
# 9. A fully connected layer mapping from (whatever dimensions we are at-- find out using .shape) to 50
# 10. A ReLU
# 11. A fully connected layer mapping from 50 to 10 dimensions
# 12. A softmax function.

# Replace this class which implements a minimal network (which still does okay)
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Valid convolution, 1 channel in, 2 channels out, stride 1, kernel size = 3
        self.conv1 = nn.Conv2d(1, 2, kernel_size=3)
        # Dropout for convolutions
        self.drop = nn.Dropout2d()
        # Fully connected layer
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x,2)
        x = F.relu(x)
        x = self.conv2(x)
        x = x.drop
        x = F.max_pool2d(x,2)
        x = F.relu(x)
        x = x.flatten(1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.log_softmax(x, dim=1)
        return x
         
        
       

## Part 2 (6 points)

Write the main training routine. It is a function that trains the `model` on the data `loader`, using the given `optimizer` and `criterion`. It should return the average loss and classification accuracy.

In [45]:
# TODO Implement the main training routine
# Return a list containing loss at each training iteration
def train(model, loader, optimizer, criterion):
    pass

In [46]:
def evaluate(model, loader, criterion):
  model.eval()
  test_loss = 0
  correct = 0

  with torch.no_grad():
    for data, target in loader:
      data = data.to(device)
      target = target.to(device)
      output = model(data)

      test_loss += criterion(output, target, size_average=False).item()
      pred = output.data.max(1, keepdim=True)[1]
      correct += (pred.squeeze() == target).sum().item()

  return test_loss / len(loader.dataset), correct / len(loader.dataset)

## Part 3 (4 points)

Define an instance of the network. Define a SGD optimizer with a learning rate of 0.01 and momentum parameter of 0.5. Define a criterion using negative log likelihood loss.

With everything defined, train the model for 10 epochs and store the returned losses and accuracies.

## Part 4 (4 points)

Generate two plots, one showing the training and test losses, and one showing the training and test accuracies, both as a function of epoch. Briefly compare the training and test results. Are the relative values what we typically expect? If not, explain why we might see lower than expected losses and/or accuracies for one or both results.

